# Notebook 10 — Final Analysis: Tables, Rankings, Metrics

Compiles all model results into:
- **Metric verification** (WIS and CWC formula checks)
- **Per-location tables** (all models ranked by CWC per dataset)
- **Overall ranking table** (averaged across 5 locations)
- **4-panel heatmaps** (PICP, MPIW, WIS, CWC)
- **Ranking bar charts**
- **Excel export** (one sheet per location + overall ranking + metric definitions)

**Metrics:**
| Metric | Formula | Better |
|---|---|---|
| PICP | mean(y ∈ [lo,hi]) × 100% | Near 95% |
| MPIW | mean(hi − lo) in metres | Lower |
| WIS | width + (10)×miss\_penalty (α=0.20) | Lower |
| CWC | MPIW × exp penalty if PICP < 95% (η=50) | Lower |

In [ ]:
# ── Cell 1: Environment + Load all results ───────────────────
RUN_ENV  = 'local'
BASE_DIR = r'C:/BMP_Data/'
if RUN_ENV == 'colab':
    from google.colab import drive; drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BMP_Data/'

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore')

SAVE_DIR = os.path.join(BASE_DIR, 'results_IndianOcean/')
TARGET_PICP = 95.0
ETA_CWC     = 50      # CWC penalty strength (Khosravi 2011, standard)
ALPHA_WIS   = 0.20    # For p10/p90 intervals

# ── Load ALL result files ──────────────────────────────────────────────────
csvs = sorted(glob.glob(os.path.join(SAVE_DIR, 'results_*.csv')))
print(f"Found {len(csvs)} result CSVs:")
for c in csvs: print(f"  {os.path.basename(c)}")

df_list = []
for c in csvs:
    df_tmp = pd.read_csv(c)
    df_list.append(df_tmp)

if not df_list:
    raise FileNotFoundError(f"No result CSVs found in {SAVE_DIR}. Run nb01-nb09 first.")

df_all = pd.concat(df_list, ignore_index=True)

# ── Add CWC column if not present ──────────────────────────────────────────
def cwc_metric(picp_pct, mpiw, target=TARGET_PICP, eta=ETA_CWC):
    """Coverage Width-based Criterion (Khosravi 2011).
    CWC = MPIW * exp(-eta*(PICP - target)) if PICP < target, else MPIW.
    Penalizes under-coverage exponentially. Perfect: CWC = MPIW at PICP = target."""
    picp_frac = picp_pct / 100.0
    tgt_frac  = target / 100.0
    if picp_frac >= tgt_frac:
        return mpiw
    return mpiw * np.exp(-eta * (picp_frac - tgt_frac))

if 'avg_cwc' not in df_all.columns:
    df_all['avg_cwc'] = df_all.apply(
        lambda r: cwc_metric(r['avg_picp'], r['avg_mpiw']), axis=1)
    df_all['std_cwc'] = df_all.get('std_mpiw', 0.0)

# ── Select best version per location+base_model ────────────────────────────
# Strip -Conf, -Cal suffixes to get base model name
df_all['base_model'] = df_all['model'].str.replace('-Cal', '').str.replace('-Conf', '')

# Priority: calibrated > original (keep calibrated if available and closer to 95%)
def pick_best(group):
    if len(group) == 1: return group.iloc[0]
    group = group.sort_values('avg_cwc')  # lower CWC = better
    return group.iloc[0]

df_best = (df_all.groupby(['location', 'base_model'])
           .apply(pick_best, include_groups=False)
           .reset_index())

print(f"\nTotal rows in df_all: {len(df_all)}")
print(f"Best per location+model: {len(df_best)}")
print("Models present:", sorted(df_best['base_model'].unique()))

In [ ]:
# ── Cell 2: Verify Winkler Score formula ────────────────────────
# WIS (Winkler Interval Score) for a (1-alpha_nominal) PI:
#
#   WIS_i = (hi_i - lo_i)
#           + (2/alpha) * max(0, lo_i - y_i)     [y below lower bound]
#           + (2/alpha) * max(0, y_i - hi_i)     [y above upper bound]
#
# For p10/p90 intervals (targeting 80% PI, alpha_nominal=0.20):
#   WIS = MPIW + (10) * mean_miss_penalty
#
# Verification:
#   When PICP = 100%: all misses = 0, so WIS = MPIW exactly. ✓
#   When PICP < 100%: WIS > MPIW (miss penalty > 0). ✓
#   WIS rewards: tight intervals + good coverage. Lower is always better.
#
# Note on alpha choice:
#   We use alpha = 0.20 (matching the trained quantile gap q_hi - q_lo = 0.90 - 0.10 = 0.80).
#   This evaluates the "quality" of the 80% PI actually trained.
#   For models that have been conformally recalibrated to 95%, the alpha is still 0.20
#   (we evaluate the interval as a 80%-target PI, and just measure how well it covers).

print("=== Winkler Score Verification on Stored Results ===")
print()
print(f"alpha = {ALPHA_WIS} (for p10/p90 intervals, targeting 80% PI)")
print(f"WIS = MPIW + (2/{ALPHA_WIS}) * mean_miss_penalty = MPIW + {2/ALPHA_WIS:.0f} * mean_miss")
print()
print("Spot checks from stored data:")
sample = df_best.dropna(subset=['avg_wis','avg_mpiw','avg_picp']).head(8)
for _, r in sample.iterrows():
    if r['avg_picp'] >= 100.0:
        expected = "WIS ≈ MPIW (100% coverage, no miss penalty)"
        ok = abs(r['avg_wis'] - r['avg_mpiw']) < 1e-4
    else:
        expected = "WIS > MPIW (some misses → penalty > 0)"
        ok = r['avg_wis'] >= r['avg_mpiw'] - 1e-6
    flag = "✓" if ok else "✗ CHECK!"
    print(f"  {r['location'][:10]:<12} {r['base_model']:<25} PICP={r['avg_picp']:6.2f}%  "
          f"MPIW={r['avg_mpiw']:.5f}  WIS={r['avg_wis']:.5f}  {flag}")

print()
print("CWC Verification:")
print(f"  CWC formula: MPIW * exp(-{ETA_CWC}*(PICP/100 - {TARGET_PICP/100})) if PICP < {TARGET_PICP}%, else MPIW")
print(f"  At PICP=95%: CWC = MPIW (no penalty)")
print(f"  At PICP=90%: CWC = MPIW * exp({ETA_CWC}*0.05) = MPIW * {np.exp(ETA_CWC*0.05):.1f}x")
print(f"  At PICP=75%: CWC = MPIW * exp({ETA_CWC}*0.20) = MPIW * {np.exp(ETA_CWC*0.20):.0f}x")

In [ ]:
# ── Cell 3: Per-dataset (per-location) tables ────────────────
# One table per location, all models ranked by CWC (lower = better).
# This is what the professor wants: per-dataset results.

LOCATIONS_ORDER = ['Arabian_Sea', 'Bay_of_Bengal', 'Andaman_Sea', 'Lakshadweep', 'South_IO']
all_tables = {}

print("=" * 110)
print("PER-LOCATION RESULTS TABLES  (ranked by CWC — lower = better)")
print("=" * 110)

for loc in LOCATIONS_ORDER:
    sub = df_best[df_best['location'] == loc].copy()
    if len(sub) == 0: continue

    # Rank by CWC
    sub = sub.sort_values('avg_cwc').reset_index(drop=True)
    sub['rank'] = range(1, len(sub)+1)
    all_tables[loc] = sub

    lat = sub['lat'].iloc[0]; lon = sub['lon'].iloc[0]
    print(f"\n{'─'*110}")
    print(f"  LOCATION: {loc}  ({lat}°N, {lon}°E)")
    print(f"{'─'*110}")
    print(f"  {'Rank':>4}  {'Model':<32}  {'PICP%':>7}  {'±Std':>5}  {'MPIW(m)':>9}  {'±Std':>7}  "
          f"{'WIS':>9}  {'CWC':>9}  {'Status':>10}")
    print(f"  {'─'*4}  {'─'*32}  {'─'*7}  {'─'*5}  {'─'*9}  {'─'*7}  {'─'*9}  {'─'*9}  {'─'*10}")

    for _, r in sub.iterrows():
        if r['avg_picp'] >= 94.0 and r['avg_picp'] <= 96.0:
            status = "✓ On target"
        elif r['avg_picp'] > 96.0:
            status = "↑ Overcov."
        elif r['avg_picp'] >= 90.0:
            status = "~ Near"
        else:
            status = "✗ Low cov."

        w_s = f"{r['avg_wis']:.5f}" if pd.notna(r.get('avg_wis')) else "     —"
        c_s = f"{r['avg_cwc']:.5f}" if pd.notna(r.get('avg_cwc')) else "     —"
        std_p = f"{r['std_picp']:.1f}" if pd.notna(r.get('std_picp')) else "  —"
        std_m = f"{r['std_mpiw']:.5f}" if pd.notna(r.get('std_mpiw')) else "      —"

        print(f"  {r['rank']:>4}  {r['base_model']:<32}  {r['avg_picp']:7.2f}  "
              f"{std_p:>5}  {r['avg_mpiw']:9.5f}  {std_m:>7}  "
              f"{w_s:>9}  {c_s:>9}  {status:>10}")

# Save per-location CSV files
for loc, sub in all_tables.items():
    fname = os.path.join(SAVE_DIR, f'table_per_location_{loc}.csv')
    sub.to_csv(fname, index=False)
print(f"\nPer-location CSVs saved to {SAVE_DIR}")

In [ ]:
# ── Cell 4: Overall ranking table (averaged across locations) ─
print("\n" + "=" * 100)
print("OVERALL MODEL RANKING  (averaged across all 5 locations)")
print("=" * 100)
print("CWC-based ranking: lower CWC = better (balances PICP ≥ 95% with tight intervals)")

ranking = df_best.groupby('base_model').agg(
    Avg_PICP      = ('avg_picp', 'mean'),
    Std_PICP      = ('std_picp', 'mean'),
    Avg_MPIW      = ('avg_mpiw', 'mean'),
    Std_MPIW      = ('std_mpiw', 'mean'),
    Avg_WIS       = ('avg_wis',  'mean'),
    Avg_CWC       = ('avg_cwc',  'mean'),
    Locations     = ('location', 'count'),
    Seeds         = ('n_seeds',  'first'),
).reset_index()

# Composite score for ranking: CWC (already accounts for coverage penalty)
ranking = ranking.sort_values('Avg_CWC').reset_index(drop=True)
ranking['Rank'] = range(1, len(ranking)+1)

# Coverage category
def coverage_cat(picp):
    if picp >= 94 and picp <= 96: return "Excellent"
    elif picp >= 92 and picp < 94: return "Good"
    elif picp > 96 and picp <= 98: return "Slight overcov."
    elif picp > 98: return "High overcov."
    elif picp >= 88 and picp < 92: return "Fair"
    else: return "Poor"

ranking['Coverage'] = ranking['Avg_PICP'].apply(coverage_cat)

print(f"\n  {'Rank':>4}  {'Model':<32}  {'AvgPICP':>8}  {'±Std':>5}  {'AvgMPIW':>9}  "
      f"{'AvgWIS':>9}  {'AvgCWC':>9}  {'Locs':>5}  {'Coverage':<16}")
print(f"  {'─'*4}  {'─'*32}  {'─'*8}  {'─'*5}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*5}  {'─'*16}")

for _, r in ranking.iterrows():
    star = " ←Best" if r['Rank'] == 1 else ""
    print(f"  {r['Rank']:>4}  {r['base_model']:<32}  {r['Avg_PICP']:8.2f}  "
          f"{r['Std_PICP']:>5.2f}  {r['Avg_MPIW']:9.5f}  "
          f"{r['Avg_WIS']:9.5f}  {r['Avg_CWC']:9.5f}  "
          f"{int(r['Locations']):>5}  {r['Coverage']:<16}{star}")

# Save ranking tables
ranking.to_csv(os.path.join(SAVE_DIR, 'table_overall_ranking.csv'), index=False)
print(f"\nSaved: table_overall_ranking.csv")

# Print top-3 summary
print(f"\n{'='*60}")
print("TOP 3 MODELS (by CWC across all locations):")
for _, r in ranking.head(3).iterrows():
    print(f"  #{r['Rank']}: {r['base_model']} — PICP={r['Avg_PICP']:.1f}%, MPIW={r['Avg_MPIW']:.5f}m, CWC={r['Avg_CWC']:.5f}")

In [ ]:
# ── Cell 5: Heatmaps — PICP, MPIW, WIS, CWC ─────────────────
LOCS = ['Arabian_Sea', 'Bay_of_Bengal', 'Andaman_Sea', 'Lakshadweep', 'South_IO']
models_ranked = ranking['base_model'].tolist()

def build_matrix(metric):
    mat = np.full((len(models_ranked), len(LOCS)), np.nan)
    for i, m in enumerate(models_ranked):
        for j, l in enumerate(LOCS):
            row = df_best[(df_best['base_model']==m) & (df_best['location']==l)]
            if len(row): mat[i,j] = float(row[metric].values[0])
    return mat

picp_m = build_matrix('avg_picp')
mpiw_m = build_matrix('avg_mpiw')
wis_m  = build_matrix('avg_wis')
cwc_m  = build_matrix('avg_cwc')

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

def draw_heatmap(ax, mat, title, cmap, fmt, vmin=None, vmax=None, annotate=True):
    im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(LOCS))); ax.set_xticklabels([l[:8] for l in LOCS], rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(models_ranked))); ax.set_yticklabels(models_ranked, fontsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if annotate:
        for i in range(len(models_ranked)):
            for j in range(len(LOCS)):
                if not np.isnan(mat[i,j]):
                    text_color = 'white' if (mat[i,j] > 0.7*np.nanmax(mat) or mat[i,j] < 0.3*np.nanmin(mat)+0.01) else 'black'
                    ax.text(j, i, fmt.format(mat[i,j]), ha='center', va='center', fontsize=7, color='black')

draw_heatmap(axes[0,0], picp_m, 'PICP (%) — Target: 94-96%\n(green=good coverage)',
             'RdYlGn', '{:.1f}', vmin=40, vmax=100)

draw_heatmap(axes[0,1], mpiw_m, 'MPIW (m) — Prediction Interval Width\n(green=narrow)',
             'RdYlGn_r', '{:.4f}')

draw_heatmap(axes[1,0], wis_m, 'Winkler Interval Score (WIS)\n(green=lower=better)',
             'RdYlGn_r', '{:.4f}')

# CWC: clip extreme values for visual clarity
cwc_clipped = np.clip(cwc_m, 0, np.nanpercentile(cwc_m, 90))
draw_heatmap(axes[1,1], cwc_clipped, 'CWC — Coverage Width Criterion\n(green=lower=better, clipped at 90th pctile)',
             'RdYlGn_r', '{:.4f}')

fig.suptitle('Indian Ocean SLA — Probabilistic Forecast Evaluation\n'
             'All Models × All Locations | 2021–2023 Daily | 80/20 Split',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fp = os.path.join(SAVE_DIR, 'heatmap_all_metrics.png')
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f"Saved: {fp}")

In [ ]:
# ── Cell 6: Ranking bar chart ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
top_n = min(12, len(ranking))  # show top 12 or all models
sub_r = ranking.head(top_n)
COLORS = plt.cm.RdYlGn(np.linspace(0.85, 0.25, top_n))

ax = axes[0,0]
bars = ax.barh(sub_r['base_model'][::-1], sub_r['Avg_PICP'][::-1], color=COLORS[::-1], alpha=0.85)
ax.axvline(94, color='gray', lw=1, linestyle=':', alpha=0.6)
ax.axvline(95, color='navy', lw=2, linestyle='--', label='Target 95%')
ax.axvline(96, color='gray', lw=1, linestyle=':', alpha=0.6)
ax.fill_betweenx([-0.5, top_n-0.5], 94, 96, alpha=0.08, color='navy')
ax.set_xlabel('Avg PICP (%)'); ax.set_title('Coverage (PICP)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)

ax = axes[0,1]
ax.barh(sub_r['base_model'][::-1], sub_r['Avg_MPIW'][::-1], color=COLORS[::-1], alpha=0.85)
ax.set_xlabel('Avg MPIW (m)'); ax.set_title('Interval Width (MPIW) — lower = better', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

ax = axes[1,0]
ax.barh(sub_r['base_model'][::-1], sub_r['Avg_WIS'][::-1], color=COLORS[::-1], alpha=0.85)
ax.set_xlabel('Avg WIS'); ax.set_title('Winkler Score (WIS) — lower = better', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

ax = axes[1,1]
cwc_clip = np.clip(sub_r['Avg_CWC'][::-1].values, 0, np.nanpercentile(sub_r['Avg_CWC'].values, 90))
ax.barh(sub_r['base_model'][::-1], cwc_clip, color=COLORS[::-1], alpha=0.85)
ax.set_xlabel('Avg CWC (clipped)'); ax.set_title('CWC Ranking — lower = better\n★ Primary ranking metric', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
# Annotate rank
for i, (_, r) in enumerate(sub_r[::-1].iterrows()):
    ax.text(0.002, i, f"#{r['Rank']}", va='center', fontsize=8, color='navy', fontweight='bold')

fig.suptitle('Indian Ocean SLA — Model Ranking (by CWC, lower = better)\n'
             '5 locations | 2021–2023 daily | p10/p90 intervals',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fp = os.path.join(SAVE_DIR, 'ranking_chart.png')
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f"Saved: {fp}")

In [ ]:
# ── Cell 7: Export final Excel report ────────────────────────
# One Excel file with multiple sheets: one per location + one for overall ranking

with pd.ExcelWriter(os.path.join(SAVE_DIR, 'Final_Results_IndianOcean.xlsx'), engine='openpyxl') as writer:
    # Sheet 1: Overall ranking
    ranking.to_excel(writer, sheet_name='Overall_Ranking', index=False)

    # Sheet 2-6: Per location
    for loc in LOCATIONS_ORDER:
        if loc in all_tables:
            sheet_name = loc[:28]   # Excel sheet name limit
            all_tables[loc].to_excel(writer, sheet_name=sheet_name, index=False)

    # Sheet 7: All results
    df_best.to_excel(writer, sheet_name='All_Results', index=False)

    # Sheet 8: Metric explanation
    meta = pd.DataFrame({
        'Metric': ['PICP', 'MPIW', 'WIS', 'CWC'],
        'Full Name': [
            'Prediction Interval Coverage Probability',
            'Mean Prediction Interval Width',
            'Winkler Interval Score',
            'Coverage Width-based Criterion'
        ],
        'Formula': [
            'mean(y in [lo,hi]) * 100%',
            'mean(hi - lo) in metres',
            f'mean(width + (2/{ALPHA_WIS})*max(0,lo-y) + (2/{ALPHA_WIS})*max(0,y-hi))',
            f'MPIW * exp(-{ETA_CWC}*(PICP/100 - {TARGET_PICP/100})) if PICP < {TARGET_PICP}%, else MPIW'
        ],
        'Better When': ['Near 95%', 'Lower', 'Lower', 'Lower'],
        'Reference': [
            'Standard PI metric',
            'Standard PI metric',
            'Winkler (1972)',
            f'Khosravi et al. (2011), eta={ETA_CWC}'
        ]
    })
    meta.to_excel(writer, sheet_name='Metric_Definitions', index=False)

print(f"Final Excel saved: Final_Results_IndianOcean.xlsx")

# Print final summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"Total models evaluated: {len(ranking)}")
print(f"Locations: {len(LOCATIONS_ORDER)}")
print(f"Target PICP: {TARGET_PICP}%  |  CWC eta: {ETA_CWC}")
print()
print("Top 3 models by CWC:")
for _, r in ranking.head(3).iterrows():
    print(f"  #{r['Rank']:2d}  {r['base_model']:<28}  PICP={r['Avg_PICP']:.1f}%  "
          f"MPIW={r['Avg_MPIW']:.5f}m  CWC={r['Avg_CWC']:.5f}")
print()
print("Note: CWC = MPIW when PICP ≥ 95%, otherwise CWC >> MPIW.")
print("Models with PICP < 90% have high CWC regardless of MPIW.")
print("Best model achieves PICP ≈ 95% with narrowest intervals.")